In [2]:
%pip install -q "transformers>=4.40,<6"

Note: you may need to restart the kernel to use updated packages.


In [3]:
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [4]:
##导入数据
train = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip",
    sep="\t",
    compression="zip"
)

test = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip",
    sep="\t",
    compression="zip"
)
##观察数据结构

print(train.shape)
print(test.shape)
print(train.head())

(25000, 3)
(25000, 2)
       id  sentiment                                             review
0  5814_8          1  With all this stuff going down at the moment w...
1  2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2  7759_3          0  The film starts with a manager (Nicholas Bell)...
3  3630_4          0  It must be assumed that those who praised this...
4  9495_8          1  Superbly trashy and wondrously unpretentious 8...


In [5]:
from transformers import AutoTokenizer, DistilBertModel

DISTILBERT_NATIVE_NAME = 'distilbert/distilbert-base-uncased'
DISTILBERT_NATIVE_MAX_LEN = 256
DISTILBERT_NATIVE_BATCH_SIZE = 8

distilbert_native_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_NATIVE_NAME)


def encode_distilbert_reviews(reviews):
    texts = [
        BeautifulSoup(str(review), 'html.parser').get_text(' ')
        for review in reviews
    ]
    encoded = distilbert_native_tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=DISTILBERT_NATIVE_MAX_LEN,
        return_token_type_ids=False,
        return_tensors='pt'
    )
    return encoded['input_ids'], encoded['attention_mask'].sum(dim=1)


distilbert_native_ids, distilbert_native_lengths = encode_distilbert_reviews(train['review'])
distilbert_native_test_ids, distilbert_native_test_lengths = encode_distilbert_reviews(test['review'])
distilbert_native_labels = torch.tensor(train['sentiment'].values, dtype=torch.float32)

distilbert_native_train_idx, distilbert_native_val_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.2,
    random_state=42,
    stratify=train['sentiment'].values
)

distilbert_native_train_dataset = TensorDataset(
    distilbert_native_ids[distilbert_native_train_idx],
    distilbert_native_lengths[distilbert_native_train_idx],
    distilbert_native_labels[distilbert_native_train_idx]
)
distilbert_native_val_dataset = TensorDataset(
    distilbert_native_ids[distilbert_native_val_idx],
    distilbert_native_lengths[distilbert_native_val_idx],
    distilbert_native_labels[distilbert_native_val_idx]
)
distilbert_native_test_dataset = TensorDataset(
    distilbert_native_test_ids,
    distilbert_native_test_lengths
)

distilbert_native_train_loader = DataLoader(
    distilbert_native_train_dataset, batch_size=DISTILBERT_NATIVE_BATCH_SIZE, shuffle=True
)
distilbert_native_val_loader = DataLoader(
    distilbert_native_val_dataset, batch_size=DISTILBERT_NATIVE_BATCH_SIZE, shuffle=False
)
distilbert_native_test_loader = DataLoader(
    distilbert_native_test_dataset, batch_size=DISTILBERT_NATIVE_BATCH_SIZE, shuffle=False
)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
class DistilBertNativeClassifier(nn.Module):

    def __init__(self, model_name=DISTILBERT_NATIVE_NAME):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        hidden_dim = self.distilbert.config.dim
        self.pre_classifier = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        positions = torch.arange(x.size(1), device=x.device)
        attention_mask = positions.unsqueeze(0) < lengths.to(x.device).unsqueeze(1)
        output = self.distilbert(
            input_ids=x,
            attention_mask=attention_mask.long(),
            return_dict=True
        )
        cls_output = output.last_hidden_state[:, 0]
        cls_output = torch.relu(self.pre_classifier(cls_output))

        return self.fc(self.dropout(cls_output)).squeeze(1)


In [7]:
criterion = nn.BCEWithLogitsLoss()


def evaluate_model(net, loader):

    net.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():

        for x, lengths, labels in loader:

            x = x.to(device)
            labels = labels.to(device)

            logits = net(x, lengths)

            loss = criterion(logits, labels)

            total_loss += loss.item() * len(labels)

            probs = torch.sigmoid(logits)

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    avg_loss = total_loss / len(loader.dataset)

    auc = roc_auc_score(
        all_labels,
        all_probs
    )

    return avg_loss, auc


def fit_model(
    net,
    train_loader,
    val_loader,
    epochs=10,
    lr=0.001,
    patience=3
):

    optimizer = torch.optim.AdamW(
        net.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    best_auc = -1
    best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):

        net.train()

        total_loss = 0

        for x, lengths, labels in train_loader:

            x = x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = net(x, lengths)

            loss = criterion(logits, labels)

            loss.backward()

            # 防止梯度过大
            torch.nn.utils.clip_grad_norm_(
                net.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item() * len(labels)

        train_loss = total_loss / len(train_loader.dataset)

        val_loss, val_auc = evaluate_model(
            net,
            val_loader
        )

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val AUC: {val_auc:.4f}"
        )

        if val_auc > best_auc:

            best_auc = val_auc

            best_state = copy.deepcopy(
                net.state_dict()
            )

            wait = 0

        else:

            wait += 1

        if wait >= patience:

            print("Early Stopping")
            break

    net.load_state_dict(best_state)

    return net, best_auc

In [8]:
def predict_test(net, test_loader):

    net.eval()

    predictions = []

    with torch.no_grad():

        for x, lengths in test_loader:

            x = x.to(device)

            logits = net(x, lengths)

            probs = torch.sigmoid(logits)

            predictions.extend(
                probs.cpu().numpy()
            )

    return np.array(predictions)


def save_submission(predictions, filename):

    submission = pd.DataFrame({
        "id": test["id"].values,
        "sentiment": predictions
    })

    # 检查提交文件
    assert len(submission) == len(test)
    assert submission["id"].is_unique
    assert submission["id"].tolist() == test["id"].tolist()

    assert np.isfinite(
        submission["sentiment"]
    ).all()

    assert submission["sentiment"].between(0, 1).all()

    path = f"/kaggle/working/{filename}"

    submission.to_csv(
        path,
        index=False
    )

    print("文件已生成：", path)
    print("文件形状：", submission.shape)
    print(submission.head())

    return submission

In [9]:
print("开始训练 DistilBERT Native")

distilbert_native_model = DistilBertNativeClassifier().to(device)

distilbert_native_model, distilbert_native_best_auc = fit_model(
    distilbert_native_model,
    distilbert_native_train_loader,
    distilbert_native_val_loader,
    epochs=3,
    lr=2e-5,
    patience=2
)

print("DistilBERT Native 最佳验证集 AUC：", distilbert_native_best_auc)

torch.save(
    distilbert_native_model.state_dict(),
    "/kaggle/working/distilbert_native_best.pth"
)
distilbert_native_predictions = predict_test(
    distilbert_native_model,
    distilbert_native_test_loader
)

distilbert_native_submission = save_submission(
    distilbert_native_predictions,
    "submission_distilbert_native.csv"
)

开始训练 DistilBERT Native


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 01 | Train Loss: 0.3326 | Val Loss: 0.3021 | Val AUC: 0.9659
Epoch 02 | Train Loss: 0.2213 | Val Loss: 0.3828 | Val AUC: 0.9685
Epoch 03 | Train Loss: 0.1361 | Val Loss: 0.4203 | Val AUC: 0.9662
DistilBERT Native 最佳验证集 AUC： 0.9684650399999999
文件已生成： /kaggle/working/submission_distilbert_native.csv
文件形状： (25000, 2)
         id  sentiment
0  12311_10   0.998486
1    8348_2   0.001530
2    5828_4   0.004904
3    7186_2   0.002760
4   12128_7   0.994512


In [10]:
%pip install -q "transformers>=4.46,<6" "accelerate>=1.0,<2"

Note: you may need to restart the kernel to use updated packages.


In [11]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
    EarlyStoppingCallback, set_seed
)

DISTILBERT_TRAINER_NAME = 'distilbert/distilbert-base-uncased'
DISTILBERT_TRAINER_MAX_LEN = 256

distilbert_trainer_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_TRAINER_NAME)


class DistilBertTrainerDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, index):
        item = {key: values[index] for key, values in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = int(self.labels[index])
        return item


def encode_distilbert_trainer_reviews(reviews):
    texts = [
        BeautifulSoup(str(review), 'html.parser').get_text(' ')
        for review in reviews
    ]
    return distilbert_trainer_tokenizer(
        texts, truncation=True, max_length=DISTILBERT_TRAINER_MAX_LEN,
        return_token_type_ids=False
    )


distilbert_trainer_train_idx, distilbert_trainer_val_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.2,
    random_state=42,
    stratify=train['sentiment'].values
)
distilbert_trainer_labels = train['sentiment'].values

distilbert_trainer_train_dataset = DistilBertTrainerDataset(
    encode_distilbert_trainer_reviews(train['review'].iloc[distilbert_trainer_train_idx]),
    distilbert_trainer_labels[distilbert_trainer_train_idx]
)
distilbert_trainer_val_dataset = DistilBertTrainerDataset(
    encode_distilbert_trainer_reviews(train['review'].iloc[distilbert_trainer_val_idx]),
    distilbert_trainer_labels[distilbert_trainer_val_idx]
)
distilbert_trainer_test_dataset = DistilBertTrainerDataset(
    encode_distilbert_trainer_reviews(test['review'])
)


In [12]:
def compute_distilbert_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
    return {'auc': roc_auc_score(labels, probabilities)}


set_seed(42)
distilbert_trainer_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_TRAINER_NAME,
    num_labels=2
)

distilbert_trainer_args = TrainingArguments(
    output_dir='/kaggle/working/distilbert_trainer_checkpoints',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=200,
    optim='adamw_torch',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='auc',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    disable_tqdm=True,
    seed=42
)

distilbert_trainer = Trainer(
    model=distilbert_trainer_model,
    args=distilbert_trainer_args,
    train_dataset=distilbert_trainer_train_dataset,
    eval_dataset=distilbert_trainer_val_dataset,
    processing_class=distilbert_trainer_tokenizer,
    data_collator=DataCollatorWithPadding(
        tokenizer=distilbert_trainer_tokenizer, pad_to_multiple_of=8
    ),
    compute_metrics=compute_distilbert_trainer_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
distilbert_trainer.train()
print('DistilBERT Trainer 最佳验证集 AUC：', distilbert_trainer.state.best_metric)

distilbert_trainer.save_model('/kaggle/working/distilbert_trainer_best')
distilbert_trainer_tokenizer.save_pretrained('/kaggle/working/distilbert_trainer_best')


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': '0.6479', 'grad_norm': '6.594', 'learning_rate': '1.409e-05', 'epoch': '1'}
{'eval_loss': '0.5204', 'eval_auc': '0.9662', 'eval_runtime': '24.94', 'eval_samples_per_second': '200.5', 'eval_steps_per_second': '12.55', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': '0.5047', 'eval_auc': '0.9701', 'eval_runtime': '25.13', 'eval_samples_per_second': '199', 'eval_steps_per_second': '12.46', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '1039', 'train_samples_per_second': '57.77', 'train_steps_per_second': '3.61', 'train_loss': '0.4233', 'epoch': '3'}
DistilBERT Trainer 最佳验证集 AUC： 0.9701495200000001


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/distilbert_trainer_best/tokenizer_config.json',
 '/kaggle/working/distilbert_trainer_best/tokenizer.json')

In [14]:
distilbert_trainer_test_output = distilbert_trainer.predict(distilbert_trainer_test_dataset)
distilbert_trainer_predictions = torch.softmax(
    torch.from_numpy(distilbert_trainer_test_output.predictions), dim=-1
).numpy()[:, 1]

distilbert_trainer_submission = save_submission(
    distilbert_trainer_predictions,
    'submission_distilbert_trainer.csv'
)


文件已生成： /kaggle/working/submission_distilbert_trainer.csv
文件形状： (25000, 2)
         id  sentiment
0  12311_10   0.996543
1    8348_2   0.002997
2    5828_4   0.107119
3    7186_2   0.011072
4   12128_7   0.969864
